# KAGAMI D-146 — TCN multi-task untuk arah harga (EKSPLORATIF, GPU)

Runtime: **GPU (T4)**. Input dibaca dari folder Drive tim (`data/processed/*`, `experiments/d146_tcn/*`); prediksi + checkpoint
ditulis ke `experiments/d146_tcn/out/` di folder Drive tim setiap refit, jadi bila sesi putus cukup jalankan ulang semua sel
(run yang sudah selesai dilewati). Seleksi varian hanya memakai origin pengembangan (target ≤ 2023); evaluasi dilakukan lokal.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
TEAM_ID = '15T5ry1VIP_ZhzXqdVAwPLI4vfRySOx8D'
cands = [f'/content/drive/.shortcut-targets-by-id/{TEAM_ID}'] + [
    os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/**/experiments/d146_tcn/tcn_mtl.py', recursive=True)]
cands = [c for c in cands if os.path.exists(os.path.join(c, 'experiments', 'd146_tcn', 'tcn_mtl.py'))]
# folder bersama tanpa pintasan: cari di dalam folder target pintasan
cands += [p for p in glob.glob('/content/drive/.shortcut-targets-by-id/*/*') if os.path.exists(os.path.join(p, 'experiments', 'd146_tcn', 'tcn_mtl.py'))]
assert cands, 'Folder tim tidak terlihat: di Google Drive, klik kanan folder tim -> Atur -> Tambahkan pintasan ke Drive Saya, lalu ulangi sel ini.'
TEAM = cands[0]; EXP = os.path.join(TEAM, 'experiments', 'd146_tcn'); OUT = os.path.join(EXP, 'out')
os.makedirs(OUT, exist_ok=True)
print('TEAM =', TEAM); print(sorted(os.listdir(EXP)))
assert os.path.exists(os.path.join(TEAM, 'data', 'processed', 'panel_weekly.parquet')), 'panel_weekly tidak ada di folder tim'

In [ ]:
!nvidia-smi -L
!rm -rf /content/kagami && mkdir -p /content/kagami && cd /content/kagami && unzip -q "$EXP/kagami_code.zip"
!python "$EXP/build_ext.py" /content/kagami "$TEAM" /content/ext.parquet

Run utama: tiga varian (deret saja, tabular saja, deret + tabular) seed 1, semua origin. ± 10–20 menit per varian di T4.

In [ ]:
for v in ['tcn', 'tab', 'tcn_tab']:
    !python "$EXP/tcn_mtl.py" --ext /content/ext.parquet --out "$OUT" --variant {v} --seed 1
print(sorted(os.listdir(OUT)))

Replikasi seed 2 & 3 (kestabilan antar seed, seperti D-144).

In [ ]:
for s in [2, 3]:
    for v in ['tcn', 'tab', 'tcn_tab']:
        !python "$EXP/tcn_mtl.py" --ext /content/ext.parquet --out "$OUT" --variant {v} --seed {s}
print(sorted(os.listdir(OUT)))